# GeoNet — Kaggle 2×T4 — sauvegarde automatique persistante

Notebook complet d'entraînement de **GeoNet** sur **2× NVIDIA T4** avec PyTorch DDP.

Cette version :

- clone/met à jour le dépôt GitHub ;
- détecte automatiquement le dataset GeoNet ;
- reprend en priorité `last_multiforme.pt` ;
- corrige les paramètres EfficientNet inutilisés avec DDP ;
- restaure les anciens checkpoints même si leur scheduler est incompatible ;
- calcule la loss en FP32 pour éviter les `Center=nan` observés en validation ;
- sauvegarde `last_multiforme.pt` à **chaque epoch** en écrasant le fichier local précédent ;
- sauvegarde `best_multiforme.pt` uniquement lorsque l'IoU validation s'améliore ;
- publie automatiquement une **nouvelle version** du dataset Kaggle `max778/checkpoints-geonet` après chaque epoch ;
- arrête l'entraînement si la publication persistante échoue après plusieurs tentatives.

Configuration par défaut :

```text
Image                  = 640×640
Batch par GPU          = 4
Nombre de GPU          = 2
Gradient accumulation  = 2
Batch global effectif  = 16
Epochs maximum         = 120
AMP                    = FP16 pour le réseau
Loss                   = FP32
Dataset checkpoints    = max778/checkpoints-geonet
```

> `/kaggle/input` est en lecture seule. Le notebook écrit d'abord les nouveaux `.pt`
> dans `/kaggle/working/checkpoints_GEONET`, puis crée automatiquement une nouvelle
> version du dataset Kaggle.


In [ ]:
# ============================================================
# 1. Vérification des 2 GPU Kaggle
# ============================================================

import os
import sys
import subprocess
from pathlib import Path
import torch

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print()
subprocess.run(["nvidia-smi"], check=False)

print()
print("CUDA disponible :", torch.cuda.is_available())
print("Nombre de GPU   :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} | {props.total_memory / 1024**3:.1f} GB")

if torch.cuda.device_count() != 2:
    print()
    print("ATTENTION : ce notebook attend 2 GPU. Active l'accélérateur 2×T4 dans Kaggle.")


In [ ]:
# ============================================================
# 2. CLONE / MISE À JOUR DU REPO GITHUB PRIVÉ
# ============================================================

from pathlib import Path
import os
import subprocess

from kaggle_secrets import UserSecretsClient

GITHUB_USERNAME = "maaxxe"
GITHUB_REPO = "Train_Kaggle_plank_Detector"

REPO_DIR = Path(
    "/kaggle/working/Train_Kaggle_plank_Detector"
)

GEONET_DIR = REPO_DIR / "GeoNet"
MODEL_FILE = GEONET_DIR / "model" / "multiforme_model.py"

print("=" * 72)
print("GITHUB - TRAIN_KAGGLE_PLANK_DETECTOR")
print("=" * 72)

# ------------------------------------------------------------
# Secret Kaggle
# ------------------------------------------------------------

token = None

try:
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("INFO : GITHUB_TOKEN non disponible dans ce notebook.")
    print("       Détail Kaggle :", exc)

# Si le dépôt est déjà présent dans la session, on peut continuer
# même sans secret.
if MODEL_FILE.exists() and not token:
    print()
    print("Repo déjà présent : utilisation de la copie existante.")
    print("GeoNet :", GEONET_DIR)

else:
    if not token:
        raise RuntimeError(
            "\nLe dépôt n'est pas présent dans /kaggle/working et "
            "le secret GITHUB_TOKEN est indisponible.\n\n"
            "Ajoute GITHUB_TOKEN dans Kaggle > Add-ons > Secrets, "
            "active-le pour ce notebook, puis relance cette cellule."
        )

    askpass = Path("/tmp/github_askpass.sh")

    askpass.write_text(
        """#!/bin/sh
case "$1" in
    *Username*) echo "$GITHUB_USERNAME" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
        encoding="utf-8",
    )
    askpass.chmod(0o700)

    env_git = os.environ.copy()
    env_git["GITHUB_USERNAME"] = GITHUB_USERNAME
    env_git["GITHUB_TOKEN"] = token
    env_git["GIT_ASKPASS"] = str(askpass)
    env_git["GIT_TERMINAL_PROMPT"] = "0"

    repo_url = (
        f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"
    )

    try:
        if (REPO_DIR / ".git").is_dir():
            print("Repo déjà cloné -> mise à jour...")

            subprocess.run(
                [
                    "git",
                    "-C",
                    str(REPO_DIR),
                    "remote",
                    "set-url",
                    "origin",
                    repo_url,
                ],
                check=True,
                env=env_git,
            )

            subprocess.run(
                [
                    "git",
                    "-C",
                    str(REPO_DIR),
                    "pull",
                    "--rebase",
                ],
                check=True,
                env=env_git,
            )

        else:
            if REPO_DIR.exists():
                raise RuntimeError(
                    f"{REPO_DIR} existe mais n'est pas un dépôt Git."
                )

            print("Clonage du repo...")

            subprocess.run(
                [
                    "git",
                    "clone",
                    repo_url,
                    str(REPO_DIR),
                ],
                check=True,
                env=env_git,
            )

    finally:
        try:
            askpass.unlink()
        except FileNotFoundError:
            pass

# ------------------------------------------------------------
# Vérification GeoNet
# ------------------------------------------------------------

print()
print("=" * 72)
print("VÉRIFICATION DU CLONE")
print("=" * 72)

print("REPO_DIR   :", REPO_DIR)
print("GEONET_DIR :", GEONET_DIR)
print("MODEL_FILE :", MODEL_FILE)

if not MODEL_FILE.is_file():
    raise FileNotFoundError(
        "\nLe clone est présent mais GeoNet/model/multiforme_model.py "
        "est introuvable."
    )

print()
print("✅ Repo GeoNet prêt.")


In [ ]:
# ============================================================
# 2B. DIAGNOSTIC RAPIDE DU REPO
# ============================================================

from pathlib import Path

working = Path("/kaggle/working")
matches = list(working.rglob("multiforme_model.py"))

print("Fichiers multiforme_model.py trouvés :")

if not matches:
    raise FileNotFoundError(
        "Aucun multiforme_model.py trouvé sous /kaggle/working."
    )

for p in matches:
    print(" -", p)

print()
print("✅ GeoNet détectable.")


## 3. Localiser le projet et les datasets

Structure utilisée par ce notebook :

```text
/kaggle/working/Train_Kaggle_plank_Detector/
└── GeoNet/
    ├── dataset/
    ├── model/
    │   └── multiforme_model.py
    ├── model_poids/
    └── train/
        └── Train_GeoNet_Kaggle_2xT4.ipynb
```

Le dossier `model/` n'a pas besoin de `__init__.py` : le script DDP importe directement `model.multiforme_model`.

Dataset d'entraînement : `/kaggle/input/geonet`  
Dataset de checkpoints : `/kaggle/input/checkpoints-geonet`


In [ ]:
# ============================================================
# 3. CHEMINS GEONET / DATASETS KAGGLE
# ============================================================

from pathlib import Path
import os
import sys


# ============================================================
# Configuration Kaggle
# ============================================================

GEONET_DATASET_SLUG = "max778/geonet"

CHECKPOINT_DATASET_SLUG = (
    "max778/checkpoints-geonet"
)

# Dossier local de travail.
# Les fichiers de ce dossier seront publiés vers Kaggle.
PUBLISH_DIR = Path(
    "/kaggle/working/checkpoints_GEONET"
)

PUBLISH_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Recherche du projet GeoNet
# ============================================================

def find_project_root():

    candidates = [
        Path(
            "/kaggle/working/"
            "Train_Kaggle_plank_Detector/"
            "GeoNet"
        ),
        Path("/kaggle/working/GeoNet"),
        Path.cwd(),
    ]

    for p in candidates:

        model_file = (
            p
            / "model"
            / "multiforme_model.py"
        )

        if model_file.is_file():

            print(
                "Projet GeoNet trouvé :",
                p.resolve()
            )

            return p.resolve()

    kaggle_working = Path(
        "/kaggle/working"
    )

    if kaggle_working.exists():

        for model_file in kaggle_working.rglob(
            "multiforme_model.py"
        ):

            if model_file.parent.name != "model":
                continue

            project = (
                model_file
                .parent
                .parent
            )

            print(
                "Projet GeoNet trouvé automatiquement :",
                project.resolve()
            )

            return project.resolve()

    raise FileNotFoundError(
        "\nProjet GeoNet introuvable.\n"
        "Fichier recherché : "
        "model/multiforme_model.py"
    )


# ============================================================
# Recherche dataset d'entraînement
# ============================================================

def find_dataset():

    kaggle_input = Path(
        "/kaggle/input"
    )

    if not kaggle_input.exists():
        raise FileNotFoundError(
            "/kaggle/input introuvable."
        )

    candidates = []

    for images_dir in kaggle_input.rglob(
        "images"
    ):

        parent = images_dir.parent

        if (
            (parent / "labels").is_dir()
        ):
            candidates.append(parent)

    geonet_candidates = [
        p
        for p in candidates
        if "geonet" in str(p).lower()
        and "checkpoint" not in str(p).lower()
    ]

    if geonet_candidates:

        return sorted(
            geonet_candidates,
            key=lambda p: len(str(p))
        )[0].resolve()

    if candidates:

        print(
            "ATTENTION : dataset GeoNet exact "
            "non identifié."
        )

        print(
            "Utilisation de :",
            candidates[0]
        )

        return candidates[0].resolve()

    raise FileNotFoundError(
        "\nDataset GeoNet introuvable.\n"
        "Aucun dossier contenant images/ + labels/ "
        "n'a été trouvé sous /kaggle/input."
    )


# ============================================================
# Recherche dataset checkpoints
# ============================================================

def find_checkpoint_dataset():

    kaggle_input = Path(
        "/kaggle/input"
    )

    preferred_names = [
        "last_multiforme.pt",
        "last_geonet.pt",
        "best_multiforme.pt",
        "best_geonet.pt",
    ]

    known_roots = [
        Path(
            "/kaggle/input/"
            "checkpoints-geonet"
        ),
        Path(
            "/kaggle/input/"
            "checkpoints_GEONET"
        ),
        Path(
            "/kaggle/input/datasets/"
            "max778/checkpoints-geonet"
        ),
        Path(
            "/kaggle/input/datasets/"
            "max778/checkpoints_GEONET"
        ),
    ]

    for root in known_roots:

        if not root.exists():
            continue

        for filename in preferred_names:

            matches = list(
                root.rglob(filename)
            )

            if matches:
                return (
                    matches[0]
                    .parent
                    .resolve()
                )

    if kaggle_input.exists():

        for filename in preferred_names:

            matches = list(
                kaggle_input.rglob(
                    filename
                )
            )

            preferred = [
                p
                for p in matches
                if "checkpoint" in str(p).lower()
                and "geonet" in str(p).lower()
            ]

            if preferred:

                return (
                    preferred[0]
                    .parent
                    .resolve()
                )

            if matches:

                return (
                    matches[0]
                    .parent
                    .resolve()
                )

    return None


# ============================================================
# Initialisation
# ============================================================

PROJECT_ROOT = find_project_root()

DATA_DIR = find_dataset()

CHECKPOINT_DATA_DIR = (
    find_checkpoint_dataset()
)

MODEL_FILE = (
    PROJECT_ROOT
    / "model"
    / "multiforme_model.py"
)

OUTPUT_DIR = PUBLISH_DIR


# ============================================================
# Python path
# ============================================================

os.chdir(
    PROJECT_ROOT
)

if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


# ============================================================
# Vérifications
# ============================================================

if not MODEL_FILE.is_file():

    raise FileNotFoundError(
        f"Modèle GeoNet introuvable : "
        f"{MODEL_FILE}"
    )

if not (
    DATA_DIR
    / "images"
).is_dir():

    raise FileNotFoundError(
        f"Dossier images introuvable : "
        f"{DATA_DIR / 'images'}"
    )

if not (
    DATA_DIR
    / "labels"
).is_dir():

    raise FileNotFoundError(
        f"Dossier labels introuvable : "
        f"{DATA_DIR / 'labels'}"
    )


images = [
    p
    for p in (
        DATA_DIR
        / "images"
    ).iterdir()
    if p.suffix.lower()
    in {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".webp",
    }
]

labels = list(
    (
        DATA_DIR
        / "labels"
    ).glob("*.txt")
)


# ============================================================
# Affichage
# ============================================================

print("=" * 72)
print("CONFIGURATION DES CHEMINS")
print("=" * 72)

print(
    "PROJECT_ROOT        :",
    PROJECT_ROOT
)

print(
    "MODEL_FILE          :",
    MODEL_FILE
)

print(
    "DATA_DIR            :",
    DATA_DIR
)

print(
    "CHECKPOINT_DATA_DIR :",
    CHECKPOINT_DATA_DIR
)

print(
    "PUBLISH_DIR         :",
    PUBLISH_DIR
)

print(
    "Dataset distant     :",
    CHECKPOINT_DATASET_SLUG
)

print()
print(
    "Images :",
    len(images)
)

print(
    "Labels :",
    len(labels)
)


if CHECKPOINT_DATA_DIR is not None:

    print()
    print(
        "Checkpoints d'entrée :"
    )

    for ckpt in sorted(
        CHECKPOINT_DATA_DIR.rglob(
            "*.pt"
        )
    ):

        print(
            " -",
            ckpt
        )


print()
print(
    "✅ Chemins GeoNet OK"
)


In [ ]:
# ============================================================
# 4. AUTHENTIFICATION + PRÉPARATION AUTO-SAVE KAGGLE
# ============================================================

import os
import shutil
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import (
    UserSecretsClient,
)


print("=" * 72)
print("AUTO-SAVE KAGGLE")
print("=" * 72)

print(
    "Dataset cible :",
    CHECKPOINT_DATASET_SLUG
)

print(
    "Dossier local :",
    PUBLISH_DIR
)


# ============================================================
# Vérifier / installer Kaggle CLI
# ============================================================

if shutil.which("kaggle") is None:

    print()
    print(
        "Kaggle CLI absent -> installation..."
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "kaggle",
        ],
        check=True,
    )


if shutil.which("kaggle") is None:

    raise RuntimeError(
        "Impossible d'installer/trouver "
        "la commande kaggle."
    )


print()
print(
    "Kaggle CLI :",
    shutil.which("kaggle")
)


# ============================================================
# Authentification
# ============================================================
#
# Priorité :
# 1. KAGGLE_API_TOKEN déjà présent
# 2. secret Kaggle KAGGLE_API_TOKEN
# 3. legacy KAGGLE_KEY (+ username max778)
# 4. credentials déjà présents dans ~/.kaggle/
#
# Aucun secret n'est affiché.
# ============================================================

secrets = UserSecretsClient()

auth_source = None


# ------------------------------------------------------------
# Nouveau token Kaggle
# ------------------------------------------------------------

if os.environ.get(
    "KAGGLE_API_TOKEN"
):

    auth_source = (
        "KAGGLE_API_TOKEN environnement"
    )

else:

    try:

        api_token = secrets.get_secret(
            "KAGGLE_API_TOKEN"
        )

    except Exception:

        api_token = None

    if api_token:

        os.environ[
            "KAGGLE_API_TOKEN"
        ] = api_token

        auth_source = (
            "Secret KAGGLE_API_TOKEN"
        )


# ------------------------------------------------------------
# Ancienne clé API Kaggle
# ------------------------------------------------------------

if auth_source is None:

    key = os.environ.get(
        "KAGGLE_KEY"
    )

    username = os.environ.get(
        "KAGGLE_USERNAME"
    )

    if not key:

        try:

            key = secrets.get_secret(
                "KAGGLE_KEY"
            )

        except Exception:

            key = None

    if not username:

        try:

            username = secrets.get_secret(
                "KAGGLE_USERNAME"
            )

        except Exception:

            username = None

    # Ton compte Kaggle.
    if not username:
        username = "max778"

    if key:

        os.environ[
            "KAGGLE_USERNAME"
        ] = username

        os.environ[
            "KAGGLE_KEY"
        ] = key

        auth_source = (
            "KAGGLE_USERNAME/KAGGLE_KEY"
        )


# ------------------------------------------------------------
# Fichiers de credentials existants
# ------------------------------------------------------------

if auth_source is None:

    access_token_file = (
        Path.home()
        / ".kaggle"
        / "access_token"
    )

    legacy_file = (
        Path.home()
        / ".kaggle"
        / "kaggle.json"
    )

    if access_token_file.is_file():

        auth_source = (
            "~/.kaggle/access_token"
        )

    elif legacy_file.is_file():

        auth_source = (
            "~/.kaggle/kaggle.json"
        )


if auth_source is None:

    raise RuntimeError(
        "\nAucune authentification Kaggle "
        "permettant de publier le dataset.\n\n"
        "Dans Kaggle > Add-ons > Secrets, "
        "ajoute de préférence :\n"
        "  KAGGLE_API_TOKEN\n\n"
        "Ou utilise les anciens secrets :\n"
        "  KAGGLE_USERNAME = max778\n"
        "  KAGGLE_KEY      = ta clé API\n\n"
        "Le notebook refuse de lancer "
        "l'entraînement tant que l'auto-save "
        "persistant n'est pas configuré."
    )


print(
    "Authentification :",
    auth_source
)


# ============================================================
# Récupérer le metadata du dataset existant
# ============================================================

PUBLISH_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

metadata_cmd = [
    "kaggle",
    "datasets",
    "metadata",
    CHECKPOINT_DATASET_SLUG,
    "-p",
    str(PUBLISH_DIR),
]

metadata_result = subprocess.run(
    metadata_cmd,
    text=True,
    capture_output=True,
)


if metadata_result.returncode != 0:

    print()
    print(
        metadata_result.stdout
    )

    print(
        metadata_result.stderr
    )

    raise RuntimeError(
        "\nImpossible d'accéder au dataset "
        f"{CHECKPOINT_DATASET_SLUG} "
        "avec les credentials Kaggle."
    )


METADATA_FILE = (
    PUBLISH_DIR
    / "dataset-metadata.json"
)


if not METADATA_FILE.is_file():

    raise FileNotFoundError(
        "dataset-metadata.json n'a pas "
        "été récupéré."
    )


# ============================================================
# Conserver le meilleur checkpoint précédent
# ============================================================
#
# Une version Kaggle remplace le contenu courant.
# Si l'epoch suivant n'est pas un nouveau BEST,
# on doit donc garder l'ancien best_multiforme.pt
# dans PUBLISH_DIR.
# ============================================================

local_best = (
    PUBLISH_DIR
    / "best_multiforme.pt"
)


if (
    not local_best.is_file()
    and CHECKPOINT_DATA_DIR is not None
):

    old_best_candidates = [
        CHECKPOINT_DATA_DIR
        / "best_multiforme.pt",

        CHECKPOINT_DATA_DIR
        / "best_geonet.pt",
    ]

    for old_best in old_best_candidates:

        if old_best.is_file():

            shutil.copy2(
                old_best,
                local_best,
            )

            print()
            print(
                "Ancien BEST copié :",
                old_best
            )

            break


# ============================================================
# Test écriture locale
# ============================================================

write_test = (
    PUBLISH_DIR
    / ".write_test"
)

write_test.write_text(
    "ok",
    encoding="utf-8",
)

write_test.unlink()


# ============================================================
# Résultat
# ============================================================

print()
print(
    "Metadata :",
    METADATA_FILE
)

print(
    "BEST local présent :",
    local_best.is_file()
)

print()
print(
    "✅ Auto-save Kaggle prêt."
)

print(
    "Chaque epoch terminée publiera une "
    "nouvelle version du dataset."
)


## 3. Configuration 2×T4

`BATCH_PER_GPU` est le batch **sur chaque T4**.

Avec les valeurs par défaut :

```text
4 × 2 GPU × accumulation 2 = batch global effectif 16
```

Si la mémoire est confortable, teste ensuite :

```python
BATCH_PER_GPU = 6
```

puis éventuellement :

```python
BATCH_PER_GPU = 8
ACCUMULATION_STEPS = 1
```


In [ ]:
# ============================================================
# 5. CONFIGURATION GEONET
# ============================================================

from pathlib import Path


IMG_SIZE = 640
EPOCHS = 120

BATCH_PER_GPU = 4
ACCUMULATION_STEPS = 2
WORKERS_PER_GPU = 2

VAL_RATIO = 0.15
SEED = 42

LR_HEAD = 1e-5
LR_BACKBONE = 5e-7
WEIGHT_DECAY = 1e-4

FREEZE_BACKBONE_EPOCHS = 2

GRAD_CLIP = 1.0
PATIENCE = 25

PRETRAINED = True
USE_AMP = True

WORLD_SIZE = 2

GLOBAL_EFFECTIVE_BATCH = (
    BATCH_PER_GPU
    * WORLD_SIZE
    * ACCUMULATION_STEPS
)


# ============================================================
# Reprise checkpoint
# ============================================================
#
# Priorité :
# 1. last_multiforme.pt local
#    (utile si on relance dans la même session)
# 2. last_multiforme.pt du dataset Kaggle
# 3. last_geonet.pt
# 4. best_multiforme.pt
# 5. best_geonet.pt
# ============================================================

def find_resume_checkpoint():

    local_last = (
        PUBLISH_DIR
        / "last_multiforme.pt"
    )

    if local_last.is_file():

        return local_last.resolve()

    if CHECKPOINT_DATA_DIR is None:

        return None

    preferred_names = [
        "last_multiforme.pt",
        "last_geonet.pt",
        "best_multiforme.pt",
        "best_geonet.pt",
    ]

    for filename in preferred_names:

        matches = list(
            CHECKPOINT_DATA_DIR.rglob(
                filename
            )
        )

        if matches:

            return (
                matches[0]
                .resolve()
            )

    return None


RESUME = find_resume_checkpoint()


if RESUME is None:

    raise FileNotFoundError(
        "\nAucun checkpoint GeoNet trouvé.\n"
        "Ce notebook est configuré pour "
        "reprendre un entraînement existant."
    )


# ============================================================
# Lire l'epoch du checkpoint
# ============================================================

checkpoint_info = torch.load(
    RESUME,
    map_location="cpu",
    weights_only=False,
)

RESUME_EPOCH = int(
    checkpoint_info.get(
        "epoch",
        checkpoint_info.get(
            "current_epoch",
            0,
        ),
    )
)

NEXT_EPOCH = (
    RESUME_EPOCH + 1
)

RESUME_BEST_IOU = checkpoint_info.get(
    "best_iou",
    checkpoint_info.get(
        "best_quality",
        None,
    ),
)

del checkpoint_info


# ============================================================
# Affichage
# ============================================================

print("=" * 72)
print("CONFIGURATION ENTRAÎNEMENT")
print("=" * 72)

print(
    "Modèle                : GeoNet"
)

print(
    "Resolution            :",
    IMG_SIZE
)

print(
    "Epoch maximum         :",
    EPOCHS
)

print(
    "Batch / GPU           :",
    BATCH_PER_GPU
)

print(
    "GPU                   :",
    WORLD_SIZE
)

print(
    "Accumulation          :",
    ACCUMULATION_STEPS
)

print(
    "Batch global effectif :",
    GLOBAL_EFFECTIVE_BATCH
)

print(
    "Workers / GPU         :",
    WORKERS_PER_GPU
)

print(
    "AMP FP16 réseau       :",
    USE_AMP
)

print(
    "Loss FP32             : True"
)

print(
    "Output local          :",
    OUTPUT_DIR
)

print(
    "Dataset distant       :",
    CHECKPOINT_DATASET_SLUG
)

print()
print(
    "Checkpoint repris     :",
    RESUME
)

print(
    "Epoch sauvegardée     :",
    RESUME_EPOCH
)

print(
    "Prochaine epoch       :",
    NEXT_EPOCH
)

print(
    "Best IoU checkpoint   :",
    RESUME_BEST_IOU
)

print()
print(
    f"✅ Reprise prévue à l'epoch "
    f"{NEXT_EPOCH}."
)


## 4. Vérifier les labels

Avec 32 points, une ligne contient normalement :

```text
1 classe + 2 coordonnées centre + 64 coordonnées contour = 67 valeurs
```

Le dataloader reste flexible et acceptera aussi un futur dataset à 64 points.


In [ ]:
# ============================================================
# 4. Vérification rapide des labels
# ============================================================

label_files = sorted((DATA_DIR / "labels").glob("*.txt"))
bad = []
empty = 0
objects = 0
point_counts = set()

for path in label_files:
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        empty += 1
        continue

    for line_no, line in enumerate(text.splitlines(), start=1):
        vals = line.split()
        if len(vals) < 7 or (len(vals) - 3) % 2 != 0:
            bad.append((path.name, line_no, len(vals)))
            continue
        objects += 1
        point_counts.add((len(vals) - 3) // 2)

print("Labels          :", len(label_files))
print("Images vides    :", empty)
print("Objets          :", objects)
print("Nb points trouvé:", sorted(point_counts))

if bad:
    print("Exemples invalides :", bad[:10])
    raise RuntimeError("Labels invalides détectés.")

print("Labels : OK")


## 6. Tester le modèle avant DDP

Ce test vérifie l'import réel de `model.multiforme_model`, la construction du réseau et un forward simple avant de lancer deux processus `torchrun`.


In [ ]:
# ============================================================
# 6. TEST DU MODÈLE GEONET AVANT DDP
# ============================================================

import gc
import sys
import torch

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Important : import direct du fichier existant dans le repo.
from model.multiforme_model import (
    IMAGENET_MEAN,
    IMAGENET_STD,
    build_model,
    multiforme_loss,
    segmentation_metrics,
)

print("=" * 72)
print("TEST IMPORT / FORWARD GEONET")
print("=" * 72)

print("MODEL_FILE      :", MODEL_FILE)
print("IMAGENET_MEAN   :", IMAGENET_MEAN)
print("IMAGENET_STD    :", IMAGENET_STD)

# pretrained=False pour que ce test ne dépende pas d'un téléchargement.
model_test = build_model(pretrained=False)

n_params = sum(p.numel() for p in model_test.parameters())
print("Paramètres      :", f"{n_params:,}")

device_test = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

model_test = model_test.to(device_test).eval()

x_test = torch.randn(
    1,
    3,
    256,
    256,
    device=device_test,
)

with torch.no_grad():
    out_test = model_test(x_test)

print("mask_logits     :", tuple(out_test["mask_logits"].shape))
print("center_logits   :", tuple(out_test["center_logits"].shape))

assert tuple(out_test["mask_logits"].shape) == (1, 1, 256, 256)
assert tuple(out_test["center_logits"].shape) == (1, 1, 128, 128)

del out_test
del x_test
del model_test
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print()
print("✅ Modèle GeoNet importé et exécuté correctement.")


## 7. Créer le script DDP

`torchrun` lance deux processus Python indépendants, un par T4. La cellule suivante crée donc :

```text
/kaggle/working/GeoNet/train_geonet_kaggle_ddp.py
```

Le script gère :

- backend NCCL ;
- un processus par GPU ;
- `DistributedSampler` ;
- AMP FP16 ;
- gradient accumulation ;
- `DDP.no_sync()` entre micro-batches ;
- réduction des métriques entre les 2 GPU ;
- sauvegarde uniquement par le `rank 0` ;
- checkpoints et reprise.


In [ ]:
DDP_SCRIPT = '# -*- coding: utf-8 -*-\n"""\nGeoNet - entraînement DDP Kaggle 2xT4.\n\nLancé par :\n    torchrun --standalone --nproc_per_node=2 train_geonet_kaggle_ddp.py ...\n\nLe --batch-per-gpu est PAR GPU.\nBatch effectif global =\n    batch_per_gpu * nombre_GPU * accumulation_steps\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport subprocess\nimport sys\nimport time\nfrom contextlib import nullcontext\nfrom pathlib import Path\nfrom typing import Dict, List, Tuple\n\nimport cv2\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import DataLoader, Dataset, Sampler, Subset\nfrom torch.utils.data.distributed import DistributedSampler\nfrom tqdm.auto import tqdm\n\nPROJECT_ROOT = Path(__file__).resolve().parent\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nfrom model.multiforme_model import (\n    IMAGENET_MEAN,\n    IMAGENET_STD,\n    build_model,\n    multiforme_loss,\n    segmentation_metrics,\n)\n\n\n# ============================================================\n# DDP\n# ============================================================\n\ndef setup_ddp():\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA indisponible : ce script DDP attend des GPU NVIDIA.")\n\n    local_rank = int(os.environ["LOCAL_RANK"])\n    rank = int(os.environ["RANK"])\n    world_size = int(os.environ["WORLD_SIZE"])\n\n    torch.cuda.set_device(local_rank)\n    dist.init_process_group(backend="nccl")\n\n    device = torch.device("cuda", local_rank)\n\n    return local_rank, rank, world_size, device\n\n\ndef cleanup_ddp():\n    if dist.is_initialized():\n        dist.destroy_process_group()\n\n\ndef is_main(rank: int) -> bool:\n    return rank == 0\n\n\ndef reduce_sums(values: Dict[str, float], device: torch.device) -> Dict[str, float]:\n    keys = list(values.keys())\n    tensor = torch.tensor(\n        [values[k] for k in keys],\n        dtype=torch.float64,\n        device=device,\n    )\n    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)\n    return {k: float(v) for k, v in zip(keys, tensor.cpu().tolist())}\n\n\n# ============================================================\n# Reproductibilité\n# ============================================================\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\n# ============================================================\n# Dataset\n# ============================================================\n\ndef draw_gaussian_max(\n    heatmap: np.ndarray,\n    cx: float,\n    cy: float,\n    sigma: float,\n) -> None:\n    h, w = heatmap.shape\n    radius = max(1, int(math.ceil(3.0 * sigma)))\n\n    x0 = max(0, int(math.floor(cx)) - radius)\n    x1 = min(w, int(math.ceil(cx)) + radius + 1)\n    y0 = max(0, int(math.floor(cy)) - radius)\n    y1 = min(h, int(math.ceil(cy)) + radius + 1)\n\n    if x0 >= x1 or y0 >= y1:\n        return\n\n    xs = np.arange(x0, x1, dtype=np.float32)\n    ys = np.arange(y0, y1, dtype=np.float32)\n    yy, xx = np.meshgrid(ys, xs, indexing="ij")\n\n    gaussian = np.exp(\n        -((xx - cx) ** 2 + (yy - cy) ** 2)\n        / (2.0 * sigma ** 2)\n    )\n\n    region = heatmap[y0:y1, x0:x1]\n    np.maximum(region, gaussian, out=region)\n\n    ix = int(round(cx))\n    iy = int(round(cy))\n    if 0 <= ix < w and 0 <= iy < h:\n        heatmap[iy, ix] = 1.0\n\n\nclass MultiFormeDataset(Dataset):\n    def __init__(self, root: str | Path, img_size: int = 512):\n        self.root = Path(root)\n        self.image_dir = self.root / "images"\n        self.label_dir = self.root / "labels"\n\n        self.img_size = int(img_size)\n        self.center_size = self.img_size // 2\n\n        if not self.image_dir.is_dir():\n            raise FileNotFoundError(f"Dossier images introuvable : {self.image_dir}")\n        if not self.label_dir.is_dir():\n            raise FileNotFoundError(f"Dossier labels introuvable : {self.label_dir}")\n\n        extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}\n        self.images = sorted(\n            p for p in self.image_dir.iterdir()\n            if p.suffix.lower() in extensions\n        )\n\n        if not self.images:\n            raise RuntimeError(f"Aucune image dans {self.image_dir}")\n\n        self.mean = np.asarray(\n            IMAGENET_MEAN,\n            dtype=np.float32,\n        ).reshape(1, 1, 3)\n\n        self.std = np.asarray(\n            IMAGENET_STD,\n            dtype=np.float32,\n        ).reshape(1, 1, 3)\n\n    def __len__(self):\n        return len(self.images)\n\n    def _read_objects(self, label_path: Path):\n        objects = []\n\n        if not label_path.exists():\n            return objects\n\n        text = label_path.read_text(encoding="utf-8").strip()\n        if not text:\n            return objects\n\n        for line_no, line in enumerate(text.splitlines(), start=1):\n            vals = line.strip().split()\n            if not vals:\n                continue\n\n            if len(vals) < 7:\n                raise ValueError(\n                    f"Label invalide {label_path}:{line_no}: "\n                    f"{len(vals)} valeurs."\n                )\n\n            cls = int(float(vals[0]))\n            cx = float(vals[1])\n            cy = float(vals[2])\n            coords = [float(v) for v in vals[3:]]\n\n            if len(coords) % 2 != 0:\n                raise ValueError(\n                    f"Nombre impair de coordonnées dans {label_path}:{line_no}"\n                )\n\n            points = np.asarray(coords, dtype=np.float32).reshape(-1, 2)\n            center = np.asarray([cx, cy], dtype=np.float32)\n\n            if points.shape[0] < 3:\n                continue\n\n            if not np.isfinite(points).all() or not np.isfinite(center).all():\n                raise ValueError(f"NaN/Inf dans {label_path}:{line_no}")\n\n            objects.append(\n                {\n                    "cls": cls,\n                    "center": np.clip(center, 0.0, 1.0),\n                    "points": np.clip(points, 0.0, 1.0),\n                }\n            )\n\n        return objects\n\n    def __getitem__(self, index: int):\n        image_path = self.images[index]\n        label_path = self.label_dir / f"{image_path.stem}.txt"\n\n        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)\n        if image_bgr is None:\n            raise RuntimeError(f"Impossible de lire {image_path}")\n\n        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)\n        image = cv2.resize(\n            image,\n            (self.img_size, self.img_size),\n            interpolation=cv2.INTER_AREA,\n        )\n\n        objects = self._read_objects(label_path)\n\n        # Pas de masks sur disque : reconstruction en RAM.\n        mask = np.zeros(\n            (self.img_size, self.img_size),\n            dtype=np.uint8,\n        )\n\n        center_heatmap = np.zeros(\n            (self.center_size, self.center_size),\n            dtype=np.float32,\n        )\n\n        for obj in objects:\n            pts_n = obj["points"]\n\n            pts_px = pts_n.copy()\n            pts_px[:, 0] *= self.img_size - 1\n            pts_px[:, 1] *= self.img_size - 1\n\n            poly = np.round(pts_px).astype(np.int32)\n            cv2.fillPoly(mask, [poly], 255)\n\n            cx_n, cy_n = obj["center"]\n            cx = float(cx_n) * (self.center_size - 1)\n            cy = float(cy_n) * (self.center_size - 1)\n\n            min_xy = pts_n.min(axis=0)\n            max_xy = pts_n.max(axis=0)\n            box_w = float(max_xy[0] - min_xy[0]) * self.center_size\n            box_h = float(max_xy[1] - min_xy[1]) * self.center_size\n\n            sigma = np.clip(\n                min(box_w, box_h) / 6.0,\n                1.5,\n                8.0,\n            )\n\n            draw_gaussian_max(\n                center_heatmap,\n                cx,\n                cy,\n                float(sigma),\n            )\n\n        image = image.astype(np.float32) / 255.0\n        image = (image - self.mean) / self.std\n        image = np.transpose(image, (2, 0, 1)).copy()\n\n        mask = (mask.astype(np.float32) / 255.0)[None, ...]\n        center_heatmap = center_heatmap[None, ...]\n\n        return {\n            "image": torch.from_numpy(image),\n            "mask": torch.from_numpy(mask),\n            "center": torch.from_numpy(center_heatmap),\n            "path": str(image_path),\n            "n_objects": len(objects),\n        }\n\n\n# ============================================================\n# Split et sampler validation sans duplication\n# ============================================================\n\ndef make_split(n: int, val_ratio: float, seed: int) -> Tuple[List[int], List[int]]:\n    if n < 2:\n        raise RuntimeError("Il faut au moins 2 images pour train/val.")\n\n    indices = list(range(n))\n    rng = random.Random(seed)\n    rng.shuffle(indices)\n\n    n_val = max(1, int(round(n * val_ratio)))\n    n_val = min(n_val, n - 1)\n\n    return indices[n_val:], indices[:n_val]\n\n\nclass DistributedEvalSampler(Sampler):\n    """Répartit la validation entre ranks sans dupliquer d\'échantillons."""\n\n    def __init__(self, dataset, num_replicas: int, rank: int):\n        self.dataset = dataset\n        self.num_replicas = num_replicas\n        self.rank = rank\n        self.indices = list(range(len(dataset)))[rank::num_replicas]\n\n    def __iter__(self):\n        return iter(self.indices)\n\n    def __len__(self):\n        return len(self.indices)\n\n\n# ============================================================\n# Epoch\n# ============================================================\n\ndef run_epoch(\n    model,\n    loader,\n    device,\n    rank: int,\n    optimizer=None,\n    scaler=None,\n    amp: bool = True,\n    grad_clip: float = 1.0,\n    accumulation_steps: int = 1,\n    train: bool = True,\n):\n    model.train(train)\n\n    sums = {\n        "loss": 0.0,\n        "bce": 0.0,\n        "dice_loss": 0.0,\n        "boundary": 0.0,\n        "center": 0.0,\n        "iou": 0.0,\n        "dice": 0.0,\n        "n_samples": 0.0,\n    }\n\n    if train:\n        optimizer.zero_grad(set_to_none=True)\n\n    iterator = loader\n    if is_main(rank):\n        iterator = tqdm(\n            loader,\n            leave=False,\n            desc="TRAIN" if train else "VAL",\n        )\n\n    n_batches = len(loader)\n\n    for step, batch in enumerate(iterator):\n        images = batch["image"].to(device, non_blocking=True)\n        masks = batch["mask"].to(device, non_blocking=True)\n        centers = batch["center"].to(device, non_blocking=True)\n\n        bs = images.shape[0]\n\n        should_step = (\n            (step + 1) % accumulation_steps == 0\n            or (step + 1) == n_batches\n        )\n\n        # Evite un all-reduce DDP sur chaque micro-batch lors de l\'accumulation.\n        sync_ctx = nullcontext()\n        if train and not should_step:\n            sync_ctx = model.no_sync()\n\n        with torch.set_grad_enabled(train):\n            with sync_ctx:\n                with torch.autocast(\n                    device_type="cuda",\n                    dtype=torch.float16,\n                    enabled=amp,\n                ):\n                    outputs = model(images)\n\n                # ------------------------------------------------\n                # Loss en FP32\n                # ------------------------------------------------\n                # Le focal center loss contient des log().\n                # En FP16, sigmoid peut saturer exactement à 1,\n                # ce qui peut provoquer log(0) -> inf/nan.\n                #\n                # Le réseau reste en AMP FP16, mais les logits\n                # utilisés pour la loss sont convertis en FP32.\n                # Le cast conserve le graphe de gradients.\n                # ------------------------------------------------\n\n                loss_outputs = {\n                    "mask_logits": outputs[\n                        "mask_logits"\n                    ].float(),\n                    "center_logits": outputs[\n                        "center_logits"\n                    ].float(),\n                }\n\n                raw_loss, parts = multiforme_loss(\n                    loss_outputs,\n                    masks.float(),\n                    centers.float(),\n                )\n\n                if not torch.isfinite(raw_loss):\n                    raise FloatingPointError(\n                        "Loss non finie détectée : "\n                        f"{float(raw_loss.detach())}"\n                    )\n\n                loss_for_backward = (\n                    raw_loss\n                    / accumulation_steps\n                )\n\n                if train:\n                    if scaler is not None and scaler.is_enabled():\n                        scaler.scale(loss_for_backward).backward()\n                    else:\n                        loss_for_backward.backward()\n\n        if train and should_step:\n            if scaler is not None and scaler.is_enabled():\n                scaler.unscale_(optimizer)\n\n            torch.nn.utils.clip_grad_norm_(\n                model.parameters(),\n                grad_clip,\n            )\n\n            if scaler is not None and scaler.is_enabled():\n                scaler.step(optimizer)\n                scaler.update()\n            else:\n                optimizer.step()\n\n            optimizer.zero_grad(set_to_none=True)\n\n        metrics = segmentation_metrics(\n            outputs["mask_logits"],\n            masks,\n        )\n\n        sums["loss"] += float(raw_loss.detach()) * bs\n        sums["bce"] += parts["bce"] * bs\n        sums["dice_loss"] += parts["dice_loss"] * bs\n        sums["boundary"] += parts["boundary"] * bs\n        sums["center"] += parts["center"] * bs\n        sums["iou"] += metrics["iou"] * bs\n        sums["dice"] += metrics["dice"] * bs\n        sums["n_samples"] += bs\n\n        if is_main(rank):\n            local_n = max(1.0, sums["n_samples"])\n            iterator.set_postfix(\n                loss=f"{sums[\'loss\'] / local_n:.4f}",\n                iou=f"{sums[\'iou\'] / local_n:.3f}",\n                dice=f"{sums[\'dice\'] / local_n:.3f}",\n            )\n\n    # Additionne les statistiques des 2 GPU.\n    sums = reduce_sums(sums, device)\n    n = max(1.0, sums.pop("n_samples"))\n\n    return {\n        key: value / n\n        for key, value in sums.items()\n    }\n\n\n# ============================================================\n# Scheduler\n# ============================================================\n\ndef make_scheduler(\n    optimizer,\n    epochs: int,\n    freeze_backbone_epochs: int,\n    min_lr_ratio: float = 0.03,\n):\n    """\n    Cosine decay.\n\n    Pour le backbone :\n    - LR = 0 pendant freeze_backbone_epochs ;\n    - puis cosine decay.\n\n    Les paramètres gardent requires_grad=True afin que DDP les enregistre\n    correctement dès sa construction.\n    """\n\n    def head_lambda(epoch_idx: int):\n        progress = min(\n            max(epoch_idx / max(1, epochs - 1), 0.0),\n            1.0,\n        )\n        return min_lr_ratio + (\n            1.0 - min_lr_ratio\n        ) * 0.5 * (1.0 + math.cos(math.pi * progress))\n\n    def backbone_lambda(epoch_idx: int):\n        if epoch_idx < freeze_backbone_epochs:\n            return 0.0\n\n        active_epochs = max(1, epochs - freeze_backbone_epochs)\n        active_idx = epoch_idx - freeze_backbone_epochs\n        progress = min(\n            max(active_idx / max(1, active_epochs - 1), 0.0),\n            1.0,\n        )\n\n        return min_lr_ratio + (\n            1.0 - min_lr_ratio\n        ) * 0.5 * (1.0 + math.cos(math.pi * progress))\n\n    return torch.optim.lr_scheduler.LambdaLR(\n        optimizer,\n        lr_lambda=[\n            head_lambda,\n            backbone_lambda,\n        ],\n    )\n\n\n# ============================================================\n# Checkpoint\n# ============================================================\n\ndef save_checkpoint(\n    path: Path,\n    model: DDP,\n    optimizer,\n    scheduler,\n    epoch: int,\n    best_iou: float,\n    history: List[Dict],\n    args,\n):\n    checkpoint = {\n        "epoch": epoch,\n        "model": model.module.state_dict(),\n        "optimizer": optimizer.state_dict(),\n        "scheduler": scheduler.state_dict(),\n        "best_iou": best_iou,\n        "history": history,\n        "config": vars(args),\n    }\n    # Écriture atomique :\n    # le checkpoint précédent reste valide tant que\n    # le nouveau fichier n\'est pas entièrement écrit.\n    tmp_path = path.with_suffix(\n        path.suffix + ".tmp"\n    )\n\n    torch.save(\n        checkpoint,\n        tmp_path,\n    )\n\n    os.replace(\n        tmp_path,\n        path,\n    )\n\n\n\n\n\n# ============================================================\n# Publication persistante sur Kaggle\n# ============================================================\n\ndef publish_kaggle_checkpoint_dataset(\n    publish_dir: Path,\n    dataset_slug: str,\n    epoch: int,\n    max_attempts: int = 3,\n) -> None:\n\n    metadata_file = (\n        publish_dir\n        / "dataset-metadata.json"\n    )\n\n    if not metadata_file.is_file():\n\n        raise FileNotFoundError(\n            "dataset-metadata.json introuvable "\n            f"dans {publish_dir}"\n        )\n\n    last_file = (\n        publish_dir\n        / "last_multiforme.pt"\n    )\n\n    if not last_file.is_file():\n\n        raise FileNotFoundError(\n            f"Checkpoint LAST introuvable : "\n            f"{last_file}"\n        )\n\n    cmd = [\n        "kaggle",\n        "datasets",\n        "version",\n        "-p",\n        str(publish_dir),\n        "-m",\n        (\n            "GeoNet auto checkpoint "\n            f"epoch {epoch}"\n        ),\n        "-q",\n        "-r",\n        "skip",\n    ]\n\n    last_error = ""\n\n    for attempt in range(\n        1,\n        max_attempts + 1,\n    ):\n\n        result = subprocess.run(\n            cmd,\n            text=True,\n            capture_output=True,\n        )\n\n        if result.returncode == 0:\n\n            print(\n                "  -> KAGGLE OK : "\n                f"epoch {epoch} persistée "\n                f"dans {dataset_slug}"\n            )\n\n            return\n\n        last_error = (\n            (result.stdout or "")\n            + "\\n"\n            + (result.stderr or "")\n        ).strip()\n\n        print(\n            "  -> ATTENTION upload Kaggle "\n            f"échoué ({attempt}/"\n            f"{max_attempts})"\n        )\n\n        if last_error:\n\n            print(\n                last_error[-2000:]\n            )\n\n        if attempt < max_attempts:\n\n            wait_seconds = (\n                15 * attempt\n            )\n\n            print(\n                "     nouvelle tentative "\n                f"dans {wait_seconds}s..."\n            )\n\n            time.sleep(\n                wait_seconds\n            )\n\n    raise RuntimeError(\n        "\\nÉchec de la sauvegarde persistante "\n        f"Kaggle après {max_attempts} tentatives.\\n"\n        "L\'entraînement est arrêté volontairement "\n        "pour éviter de continuer sans checkpoint "\n        "persistant.\\n\\n"\n        f"Dernière erreur :\\n{last_error}"\n    )\n\n\n# ============================================================\n# EfficientNetV2-S : couche finale inutilisée par GeoNet\n# ============================================================\n\ndef disable_unused_efficientnet_tail(model) -> None:\n    """\n    GeoNet utilise encoder.features[0] ... encoder.features[6].\n    torchvision EfficientNetV2-S possède aussi features[7], mais cette\n    couche n\'est jamais appelée dans MultiFormeNet.forward().\n\n    On conserve la couche dans le modèle pour rester compatible avec\n    les anciens checkpoints, mais on désactive ses gradients afin que\n    DDP n\'attende pas de réduction pour ces paramètres inutilisés.\n    """\n    features = getattr(model.encoder, "features", None)\n\n    if features is None:\n        return\n\n    if len(features) <= 7:\n        return\n\n    for p in features[7].parameters():\n        p.requires_grad = False\n\n\n# ============================================================\n# Main\n# ============================================================\n\ndef parse_args():\n    parser = argparse.ArgumentParser()\n\n    parser.add_argument("--data", type=str, required=True)\n    parser.add_argument("--output", type=str, default="runs/geonet_kaggle")\n\n    parser.add_argument("--img-size", type=int, default=512)\n    parser.add_argument("--epochs", type=int, default=120)\n\n    # IMPORTANT : batch PAR GPU.\n    parser.add_argument("--batch-per-gpu", type=int, default=4)\n    parser.add_argument("--accumulation", type=int, default=2)\n\n    # workers PAR GPU.\n    parser.add_argument("--workers", type=int, default=2)\n\n    parser.add_argument("--val-ratio", type=float, default=0.15)\n    parser.add_argument("--seed", type=int, default=42)\n\n    parser.add_argument("--lr-head", type=float, default=3e-4)\n    parser.add_argument("--lr-backbone", type=float, default=3e-5)\n    parser.add_argument("--weight-decay", type=float, default=1e-4)\n\n    parser.add_argument("--freeze-backbone-epochs", type=int, default=2)\n    parser.add_argument("--grad-clip", type=float, default=1.0)\n    parser.add_argument("--patience", type=int, default=25)\n\n    parser.add_argument("--resume", type=str, default="")\n\n    parser.add_argument(\n        "--publish-dir",\n        type=str,\n        required=True,\n    )\n\n    parser.add_argument(\n        "--kaggle-dataset",\n        type=str,\n        required=True,\n    )\n\n    parser.add_argument("--no-pretrained", action="store_true")\n    parser.add_argument("--no-amp", action="store_true")\n\n    return parser.parse_args()\n\n\ndef main():\n    args = parse_args()\n\n    local_rank, rank, world_size, device = setup_ddp()\n\n    torch.backends.cudnn.benchmark = True\n\n    # Split identique sur tous les ranks.\n    seed_everything(args.seed + rank)\n\n    output_dir = Path(args.output)\n    publish_dir = Path(args.publish_dir)\n\n    if is_main(rank):\n        output_dir.mkdir(\n            parents=True,\n            exist_ok=True,\n        )\n\n        publish_dir.mkdir(\n            parents=True,\n            exist_ok=True,\n        )\n\n        metadata_file = (\n            publish_dir\n            / "dataset-metadata.json"\n        )\n\n        if not metadata_file.is_file():\n            raise FileNotFoundError(\n                "Metadata Kaggle manquant : "\n                f"{metadata_file}"\n            )\n\n    dist.barrier()\n\n    dataset = MultiFormeDataset(\n        args.data,\n        img_size=args.img_size,\n    )\n\n    train_idx, val_idx = make_split(\n        len(dataset),\n        args.val_ratio,\n        args.seed,\n    )\n\n    train_set = Subset(dataset, train_idx)\n    val_set = Subset(dataset, val_idx)\n\n    train_sampler = DistributedSampler(\n        train_set,\n        num_replicas=world_size,\n        rank=rank,\n        shuffle=True,\n        seed=args.seed,\n        drop_last=False,\n    )\n\n    val_sampler = DistributedEvalSampler(\n        val_set,\n        num_replicas=world_size,\n        rank=rank,\n    )\n\n    train_loader = DataLoader(\n        train_set,\n        batch_size=args.batch_per_gpu,\n        sampler=train_sampler,\n        num_workers=args.workers,\n        pin_memory=True,\n        persistent_workers=args.workers > 0,\n        drop_last=False,\n    )\n\n    val_loader = DataLoader(\n        val_set,\n        batch_size=args.batch_per_gpu,\n        sampler=val_sampler,\n        num_workers=args.workers,\n        pin_memory=True,\n        persistent_workers=args.workers > 0,\n        drop_last=False,\n    )\n\n    if is_main(rank):\n        print("=" * 78)\n        print("MULTIFORMENET - KAGGLE DDP")\n        print("=" * 78)\n        print(f"GPU                 : {world_size}")\n        print(f"Train / Val         : {len(train_set)} / {len(val_set)}")\n        print(f"Image size          : {args.img_size}")\n        print(f"Batch / GPU         : {args.batch_per_gpu}")\n        print(f"Accumulation        : x{args.accumulation}")\n        print(\n            "Batch global effectif: "\n            f"{args.batch_per_gpu * world_size * args.accumulation}"\n        )\n        print(f"Workers / GPU       : {args.workers}")\n        print(f"AMP FP16            : {not args.no_amp}")\n        print(f"Pretrained          : {not args.no_pretrained}")\n        print("=" * 78)\n\n        if len(dataset) < 100:\n            print(\n                "ATTENTION : dataset très petit. "\n                "Le code fonctionnera mais ce n\'est pas suffisant pour entraîner le modèle."\n            )\n\n    # Rank 0 charge/télécharge d\'abord les poids ImageNet.\n    pretrained = not args.no_pretrained\n\n    pretrained_for_build = (\n        pretrained\n        and not bool(args.resume)\n    )\n\n    if is_main(rank):\n        model = build_model(\n            pretrained=pretrained_for_build\n        )\n\n    dist.barrier()\n\n    if not is_main(rank):\n        model = build_model(\n            pretrained=pretrained_for_build\n        )\n\n    dist.barrier()\n\n    # Important pour DDP :\n    # features[7] est enregistré dans EfficientNetV2-S mais n\'est\n    # jamais utilisé par le forward de GeoNet.\n    disable_unused_efficientnet_tail(model)\n\n    model = model.to(device)\n\n    # DDP doit voir tous les paramètres dès le départ.\n    model = DDP(\n        model,\n        device_ids=[local_rank],\n        output_device=local_rank,\n        broadcast_buffers=False,\n        find_unused_parameters=False,\n    )\n\n    head_params = [\n        p\n        for name, p in model.module.named_parameters()\n        if not name.startswith("encoder.")\n    ]\n\n    backbone_params = list(\n        model.module.encoder.parameters()\n    )\n\n    optimizer = torch.optim.AdamW(\n        [\n            {\n                "params": head_params,\n                "lr": args.lr_head,\n            },\n            {\n                "params": backbone_params,\n                "lr": args.lr_backbone,\n            },\n        ],\n        weight_decay=args.weight_decay,\n    )\n\n    scheduler = make_scheduler(\n        optimizer,\n        epochs=args.epochs,\n        freeze_backbone_epochs=args.freeze_backbone_epochs,\n    )\n\n    amp = not args.no_amp\n    scaler = torch.amp.GradScaler(\n        "cuda",\n        enabled=amp,\n    )\n\n    start_epoch = 1\n    best_iou = -1.0\n    history = []\n\n    if args.resume:\n        checkpoint = torch.load(\n            args.resume,\n            map_location=device,\n            weights_only=False,\n        )\n\n        model_state = checkpoint.get(\n            "model",\n            checkpoint.get(\n                "model_state_dict",\n                checkpoint,\n            ),\n        )\n\n        model.module.load_state_dict(\n            model_state,\n            strict=True,\n        )\n\n        if "optimizer" in checkpoint:\n            optimizer.load_state_dict(\n                checkpoint["optimizer"]\n            )\n        elif "optimizer_state_dict" in checkpoint:\n            optimizer.load_state_dict(\n                checkpoint["optimizer_state_dict"]\n            )\n        elif is_main(rank):\n            print(\n                "ATTENTION : état optimizer absent du checkpoint ; "\n                "les poids sont repris mais l\'optimizer repart neuf."\n            )\n\n        checkpoint_epoch = int(\n            checkpoint.get(\n                "epoch",\n                checkpoint.get(\n                    "current_epoch",\n                    0,\n                ),\n            )\n        )\n\n        start_epoch = (\n            checkpoint_epoch + 1\n        )\n\n        scheduler_state = checkpoint.get(\n            "scheduler",\n            checkpoint.get(\n                "scheduler_state_dict"\n            ),\n        )\n\n        scheduler_restored = False\n\n        if scheduler_state is not None:\n\n            try:\n\n                scheduler.load_state_dict(\n                    dict(scheduler_state)\n                )\n\n                scheduler_restored = True\n\n                if is_main(rank):\n\n                    print(\n                        "Scheduler restauré "\n                        "depuis le checkpoint."\n                    )\n\n            except Exception as exc:\n\n                if is_main(rank):\n\n                    print()\n                    print(\n                        "ATTENTION : scheduler "\n                        "du checkpoint incompatible."\n                    )\n\n                    print(\n                        "Erreur :",\n                        repr(exc)\n                    )\n\n        if not scheduler_restored:\n\n            base_lrs = [\n                float(args.lr_head),\n                float(args.lr_backbone),\n            ]\n\n            scheduler.base_lrs = (\n                base_lrs\n            )\n\n            scheduler.last_epoch = (\n                checkpoint_epoch\n            )\n\n            scheduler._step_count = (\n                checkpoint_epoch + 1\n            )\n\n            resumed_lrs = []\n\n            for (\n                param_group,\n                base_lr,\n                lr_lambda,\n            ) in zip(\n                optimizer.param_groups,\n                base_lrs,\n                scheduler.lr_lambdas,\n            ):\n\n                param_group[\n                    "initial_lr"\n                ] = base_lr\n\n                lr = float(\n                    base_lr\n                    * lr_lambda(\n                        checkpoint_epoch\n                    )\n                )\n\n                param_group[\n                    "lr"\n                ] = lr\n\n                resumed_lrs.append(\n                    lr\n                )\n\n            scheduler._last_lr = (\n                resumed_lrs\n            )\n\n            if is_main(rank):\n\n                print(\n                    "Scheduler reconstruit :"\n                )\n\n                print(\n                    "  checkpoint epoch :",\n                    checkpoint_epoch\n                )\n\n                print(\n                    "  reprise epoch     :",\n                    start_epoch\n                )\n\n                print(\n                    "  LR head           :",\n                    f"{resumed_lrs[0]:.3e}"\n                )\n\n                print(\n                    "  LR backbone       :",\n                    f"{resumed_lrs[1]:.3e}"\n                )\n        best_iou = float(\n            checkpoint.get("best_iou", -1.0)\n        )\n        history = list(\n            checkpoint.get("history", [])\n        )\n\n        if is_main(rank):\n            print(\n                f"Reprise depuis {args.resume} "\n                f"-> epoch {start_epoch}"\n            )\n\n    dist.barrier()\n\n    epochs_without_improvement = 0\n    training_start = time.time()\n\n    for epoch in range(start_epoch, args.epochs + 1):\n        train_sampler.set_epoch(epoch)\n\n        train_stats = run_epoch(\n            model=model,\n            loader=train_loader,\n            device=device,\n            rank=rank,\n            optimizer=optimizer,\n            scaler=scaler,\n            amp=amp,\n            grad_clip=args.grad_clip,\n            accumulation_steps=args.accumulation,\n            train=True,\n        )\n\n        with torch.no_grad():\n            val_stats = run_epoch(\n                model=model,\n                loader=val_loader,\n                device=device,\n                rank=rank,\n                optimizer=None,\n                scaler=None,\n                amp=amp,\n                grad_clip=args.grad_clip,\n                accumulation_steps=1,\n                train=False,\n            )\n\n        # Tous les ranks appellent le scheduler.\n        scheduler.step()\n\n        improved = val_stats["iou"] > best_iou\n\n        if improved:\n            best_iou = val_stats["iou"]\n            epochs_without_improvement = 0\n        else:\n            epochs_without_improvement += 1\n\n        if is_main(rank):\n            elapsed_min = (\n                time.time() - training_start\n            ) / 60.0\n\n            record = {\n                "epoch": epoch,\n                "train": train_stats,\n                "val": val_stats,\n                "lr_head": optimizer.param_groups[0]["lr"],\n                "lr_backbone": optimizer.param_groups[1]["lr"],\n            }\n            history.append(record)\n\n            print(\n                f"[{epoch:03d}/{args.epochs:03d}] "\n                f"train={train_stats[\'loss\']:.4f} "\n                f"IoU={train_stats[\'iou\']:.4f} "\n                f"| val={val_stats[\'loss\']:.4f} "\n                f"IoU={val_stats[\'iou\']:.4f} "\n                f"Dice={val_stats[\'dice\']:.4f} "\n                f"Center={val_stats[\'center\']:.4f} "\n                f"| LR={optimizer.param_groups[0][\'lr\']:.2e}/"\n                f"{optimizer.param_groups[1][\'lr\']:.2e} "\n                f"| {elapsed_min:.1f} min"\n            )\n\n            # ================================================\n            # LAST : écrasé à chaque epoch\n            # ================================================\n\n            last_path = (\n                publish_dir\n                / "last_multiforme.pt"\n            )\n\n            save_checkpoint(\n                last_path,\n                model,\n                optimizer,\n                scheduler,\n                epoch,\n                best_iou,\n                history,\n                args,\n            )\n\n            print(\n                "  -> LAST : "\n                f"epoch {epoch} sauvegardée"\n            )\n\n            # ================================================\n            # BEST : écrasé seulement si meilleur IoU\n            # ================================================\n\n            if improved:\n\n                best_path = (\n                    publish_dir\n                    / "best_multiforme.pt"\n                )\n\n                save_checkpoint(\n                    best_path,\n                    model,\n                    optimizer,\n                    scheduler,\n                    epoch,\n                    best_iou,\n                    history,\n                    args,\n                )\n\n                print(\n                    f"  -> BEST : "\n                    f"IoU={best_iou:.4f}"\n                )\n\n            # ================================================\n            # Historique\n            # ================================================\n\n            (\n                publish_dir\n                / "history.json"\n            ).write_text(\n                json.dumps(\n                    history,\n                    indent=2\n                ),\n                encoding="utf-8",\n            )\n\n            # ================================================\n            # Publication persistante Kaggle\n            # ================================================\n\n            publish_kaggle_checkpoint_dataset(\n                publish_dir=publish_dir,\n                dataset_slug=(\n                    args.kaggle_dataset\n                ),\n                epoch=epoch,\n            )\n\n        # Tous les ranks prennent la même décision d\'arrêt.\n        stop_tensor = torch.tensor(\n            [\n                1\n                if epochs_without_improvement >= args.patience\n                else 0\n            ],\n            device=device,\n            dtype=torch.int32,\n        )\n        dist.broadcast(stop_tensor, src=0)\n\n        if int(stop_tensor.item()) == 1:\n            if is_main(rank):\n                print(\n                    f"Early stopping : {args.patience} epochs "\n                    "sans amélioration de l\'IoU."\n                )\n            break\n\n        dist.barrier()\n\n    if is_main(rank):\n        print("=" * 78)\n        print(f"Terminé. Best IoU = {best_iou:.4f}")\n        print(\n            "LAST checkpoint :",\n            publish_dir\n            / "last_multiforme.pt",\n        )\n\n        print(\n            "BEST checkpoint :",\n            publish_dir\n            / "best_multiforme.pt",\n        )\n\n        print(\n            "Dataset Kaggle :",\n            args.kaggle_dataset,\n        )\n        print("=" * 78)\n\n    cleanup_ddp()\n\n\nif __name__ == "__main__":\n    main()\n'

DDP_PATH = PROJECT_ROOT / 'train_geonet_kaggle_ddp.py'
DDP_PATH.write_text(DDP_SCRIPT, encoding='utf-8')

import ast
ast.parse(DDP_SCRIPT)

print('Script créé :', DDP_PATH)
print('Syntaxe      : OK')
print('Auto-save    : Kaggle à chaque epoch')
print('Loss         : FP32 (réseau AMP FP16)')


## 8. Lancer l'entraînement sur les 2 T4

À chaque epoch complètement terminée :

```text
/kaggle/working/checkpoints_GEONET/
├── last_multiforme.pt   <- écrasé à chaque epoch
├── best_multiforme.pt   <- remplacé uniquement si meilleur IoU
├── history.json
└── dataset-metadata.json
```

Puis le notebook publie automatiquement une nouvelle version de :

```text
max778/checkpoints-geonet
```

L'epoch suivante ne démarre qu'après confirmation de la sauvegarde persistante Kaggle.


In [ ]:
# ============================================================
# 8. TRAIN 2×T4 - TORCHRUN + AUTO-SAVE KAGGLE
# ============================================================

import os
import shlex
import subprocess
import sys
from pathlib import Path

import torch


# ============================================================
# Chemins
# ============================================================

DDP_PATH = Path(
    DDP_PATH
)

PROJECT_ROOT = Path(
    PROJECT_ROOT
)

DATA_DIR = Path(
    DATA_DIR
)

OUTPUT_DIR = Path(
    OUTPUT_DIR
)

PUBLISH_DIR = Path(
    PUBLISH_DIR
)

RESUME = Path(
    RESUME
)


# ============================================================
# Vérifications
# ============================================================

print("=" * 72)
print("VÉRIFICATIONS AVANT TORCHRUN")
print("=" * 72)

print(
    "PROJECT_ROOT :",
    PROJECT_ROOT
)

print(
    "DATA_DIR     :",
    DATA_DIR
)

print(
    "DDP_PATH     :",
    DDP_PATH
)

print(
    "OUTPUT_DIR   :",
    OUTPUT_DIR
)

print(
    "PUBLISH_DIR  :",
    PUBLISH_DIR
)

print(
    "CHECKPOINT   :",
    RESUME
)

print(
    "DATASET      :",
    CHECKPOINT_DATASET_SLUG
)

print()
print(
    "Projet existe :",
    PROJECT_ROOT.is_dir()
)

print(
    "Dataset existe:",
    DATA_DIR.is_dir()
)

print(
    "Script existe :",
    DDP_PATH.is_file()
)

print(
    "Checkpoint    :",
    RESUME.is_file()
)

print(
    "Metadata      :",
    (
        PUBLISH_DIR
        / "dataset-metadata.json"
    ).is_file()
)

print(
    "CUDA          :",
    torch.cuda.is_available()
)

print(
    "GPU détectés  :",
    torch.cuda.device_count()
)


for i in range(
    torch.cuda.device_count()
):

    print(
        f"GPU {i}          :",
        torch.cuda.get_device_name(i)
    )


# ============================================================
# Sécurité
# ============================================================

if not PROJECT_ROOT.is_dir():

    raise FileNotFoundError(
        PROJECT_ROOT
    )

if not DATA_DIR.is_dir():

    raise FileNotFoundError(
        DATA_DIR
    )

if not DDP_PATH.is_file():

    raise FileNotFoundError(
        f"Script DDP introuvable : "
        f"{DDP_PATH}"
    )

if not RESUME.is_file():

    raise FileNotFoundError(
        f"Checkpoint introuvable : "
        f"{RESUME}"
    )

if not (
    PUBLISH_DIR
    / "dataset-metadata.json"
).is_file():

    raise FileNotFoundError(
        "dataset-metadata.json absent. "
        "Relance la cellule AUTO-SAVE KAGGLE."
    )

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA n'est pas disponible."
    )

if torch.cuda.device_count() != 2:

    raise RuntimeError(
        "Ce notebook attend exactement "
        "2 GPU T4, mais "
        f"{torch.cuda.device_count()} "
        "GPU sont détectés."
    )


# ============================================================
# Commande DDP
# ============================================================

cmd = [
    sys.executable,
    "-m",
    "torch.distributed.run",

    "--standalone",
    "--nproc_per_node=2",

    str(DDP_PATH),

    "--data",
    str(DATA_DIR),

    "--output",
    str(OUTPUT_DIR),

    "--publish-dir",
    str(PUBLISH_DIR),

    "--kaggle-dataset",
    CHECKPOINT_DATASET_SLUG,

    "--img-size",
    str(IMG_SIZE),

    "--epochs",
    str(EPOCHS),

    "--batch-per-gpu",
    str(BATCH_PER_GPU),

    "--accumulation",
    str(ACCUMULATION_STEPS),

    "--workers",
    str(WORKERS_PER_GPU),

    "--val-ratio",
    str(VAL_RATIO),

    "--seed",
    str(SEED),

    "--lr-head",
    str(LR_HEAD),

    "--lr-backbone",
    str(LR_BACKBONE),

    "--weight-decay",
    str(WEIGHT_DECAY),

    "--freeze-backbone-epochs",
    str(FREEZE_BACKBONE_EPOCHS),

    "--grad-clip",
    str(GRAD_CLIP),

    "--patience",
    str(PATIENCE),

    "--resume",
    str(RESUME),
]


if not PRETRAINED:

    cmd.append(
        "--no-pretrained"
    )

if not USE_AMP:

    cmd.append(
        "--no-amp"
    )


# ============================================================
# Environnement
# ============================================================

env = os.environ.copy()

env[
    "OMP_NUM_THREADS"
] = "2"

env[
    "NCCL_DEBUG"
] = "WARN"

env[
    "PYTHONUNBUFFERED"
] = "1"


old_pythonpath = env.get(
    "PYTHONPATH",
    "",
)

if old_pythonpath:

    env["PYTHONPATH"] = (
        str(PROJECT_ROOT)
        + os.pathsep
        + old_pythonpath
    )

else:

    env["PYTHONPATH"] = (
        str(PROJECT_ROOT)
    )


# ============================================================
# Vérifier que les credentials sont transmis au subprocess
# ============================================================

has_token = bool(
    env.get(
        "KAGGLE_API_TOKEN"
    )
)

has_legacy = bool(
    env.get(
        "KAGGLE_KEY"
    )
)

has_file_auth = (
    (
        Path.home()
        / ".kaggle"
        / "access_token"
    ).is_file()
    or
    (
        Path.home()
        / ".kaggle"
        / "kaggle.json"
    ).is_file()
)

if not (
    has_token
    or has_legacy
    or has_file_auth
):

    raise RuntimeError(
        "Credentials Kaggle non disponibles "
        "dans l'environnement torchrun."
    )


# ============================================================
# Affichage
# ============================================================

print()
print("=" * 72)
print("COMMANDE TORCHRUN")
print("=" * 72)

print(
    shlex.join(cmd)
)

print()
print(
    "Reprise checkpoint :",
    RESUME
)

print(
    "Publication        :",
    CHECKPOINT_DATASET_SLUG
)

print(
    "Mode sauvegarde    : "
    "chaque epoch"
)

print()
print("=" * 72)
print("DÉBUT ENTRAÎNEMENT")
print("=" * 72)
print()


# ============================================================
# Lancement
# ============================================================
#
# stdout/stderr ne sont PAS capturés :
# tqdm reste sur une seule ligne dans Kaggle.
# ============================================================

try:

    result = subprocess.run(
        cmd,
        cwd=str(PROJECT_ROOT),
        env=env,
        check=False,
    )

except KeyboardInterrupt:

    print()
    print()
    print("=" * 72)
    print(
        "ENTRAÎNEMENT INTERROMPU"
    )
    print("=" * 72)

    print(
        "Le dernier epoch complètement "
        "terminé a déjà été sauvegardé "
        "et publié sur Kaggle."
    )

    raise


# ============================================================
# Résultat
# ============================================================

print()
print()
print("=" * 72)
print("FIN TORCHRUN")
print("=" * 72)

print(
    "Code retour :",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "\nTorchrun a échoué avec le code "
        f"{result.returncode}.\n\n"
        "L'erreur complète est affichée "
        "juste au-dessus.\n"
        "Si l'erreur vient de l'upload Kaggle, "
        "le dernier .pt est toujours présent "
        "dans /kaggle/working/checkpoints_GEONET."
    )


print()
print(
    "✅ Entraînement GeoNet terminé."
)

print(
    "✅ Derniers checkpoints publiés dans :",
    CHECKPOINT_DATASET_SLUG
)


In [ ]:
# ============================================================
# 9. VÉRIFIER LES CHECKPOINTS SAUVEGARDÉS
# ============================================================

from pathlib import Path
import subprocess
import torch


print("=" * 72)
print("CHECKPOINTS LOCAUX")
print("=" * 72)

for name in [
    "last_multiforme.pt",
    "best_multiforme.pt",
]:

    path = (
        PUBLISH_DIR
        / name
    )

    if not path.is_file():

        print(
            f"❌ {name} introuvable"
        )

        continue

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    epoch = int(
        ckpt.get(
            "epoch",
            ckpt.get(
                "current_epoch",
                0,
            ),
        )
    )

    best_iou = ckpt.get(
        "best_iou",
        None,
    )

    size_mb = (
        path.stat().st_size
        / 1024**2
    )

    print()
    print(
        f"✅ {name}"
    )

    print(
        "   Epoch    :",
        epoch
    )

    print(
        "   Prochaine:",
        epoch + 1
    )

    print(
        "   Best IoU :",
        best_iou
    )

    print(
        "   Taille   :",
        f"{size_mb:.2f} MB"
    )


print()
print("=" * 72)
print("VERSION DISTANTE KAGGLE")
print("=" * 72)

result = subprocess.run(
    [
        "kaggle",
        "datasets",
        "status",
        CHECKPOINT_DATASET_SLUG,
    ],
    text=True,
    capture_output=True,
)

print(
    result.stdout
    or result.stderr
)

if result.returncode != 0:

    raise RuntimeError(
        "Impossible de lire le statut "
        "du dataset Kaggle."
    )


## 9. Vérifier la VRAM

Le nouveau modèle est plus lourd que PlankEye. Commence à `4/GPU`.

Si les deux T4 restent loin de leur limite mémoire, essaie `6/GPU`, puis `8/GPU`.
Un batch physique plus grand est généralement plus rapide qu'une accumulation de gradients.


In [ ]:
# ============================================================
# 7. GPU / VRAM
# ============================================================

subprocess.run(["nvidia-smi"], check=False)


## 10. Courbes après entraînement

Affichage de la loss, de l'IoU et du Dice.


In [ ]:
# ============================================================
# 8. Graphiques
# ============================================================

import json
import matplotlib.pyplot as plt

history_path = OUTPUT_DIR / "history.json"

if not history_path.exists():
    print("Pas encore de history.json :", history_path)
else:
    history = json.loads(history_path.read_text(encoding="utf-8"))
    epochs_plot = [r["epoch"] for r in history]

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_plot, [r["train"]["loss"] for r in history], label="Train loss")
    plt.plot(epochs_plot, [r["val"]["loss"] for r in history], label="Val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("GeoNet — Loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_plot, [r["train"]["iou"] for r in history], label="Train IoU")
    plt.plot(epochs_plot, [r["val"]["iou"] for r in history], label="Val IoU")
    plt.xlabel("Epoch")
    plt.ylabel("IoU")
    plt.title("GeoNet — IoU")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_plot, [r["val"]["dice"] for r in history], label="Val Dice")
    plt.xlabel("Epoch")
    plt.ylabel("Dice")
    plt.title("GeoNet — Dice validation")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    best = max(history, key=lambda r: r["val"]["iou"])
    print(
        f"Best epoch : {best['epoch']} | "
        f"IoU={best['val']['iou']:.4f} | "
        f"Dice={best['val']['dice']:.4f}"
    )


## Reprise automatique

Au prochain démarrage Kaggle, le notebook cherche en priorité :

```text
last_multiforme.pt
last_geonet.pt
best_multiforme.pt
best_geonet.pt
```

Si `last_multiforme.pt` est à l'epoch 37, l'entraînement reprend à l'epoch 38.

Pendant une même session, un `last_multiforme.pt` déjà présent dans
`/kaggle/working/checkpoints_GEONET` est prioritaire sur l'ancien fichier monté
dans `/kaggle/input`.
